In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from source_code.itransformer_dataset import load_and_preprocess_data
import pandas as pd

In [ ]:
# --- Step A: Load Data ---
# 1. Load Data
(
    df_1,
    df_all,
    hist_exog_cols_1,
    time_exog_cols,
    target_col_1,
    all_target,
) = load_and_preprocess_data('/workspaces/Max-Plank-Weather-using-iTransformer-PatchTST-DLinear-N-HiTS-dan-GRU/max_planck_weather_ts.csv')

In [ ]:
# --- Step B: Train-Test Split (Long Format) ---
HORIZON = 144  # Prediksi 144 jam ke depan
INPUT_SIZE = 288  # Lookback 288 jam ke belakang

NUM_TIMESTAMPS = 10000  # Total timestamp yang digunakan
TEST_TIMESTAMPS = 2000  # Timestamp untuk pengujian

# =====================================================================
# SKENARIO 1: DATASET 1 TARGET (Suhu T)
# =====================================================================
NUM_TARGETS_1 = len(target_col_1)  # 1 Target

# Ambil 10.000 timestamp terakhir berbasis kolom 'ds'
unique_ds_1 = df_1['ds'].unique()[-NUM_TIMESTAMPS:]
df_sample_1 = df_1[df_1['ds'].isin(unique_ds_1)].reset_index(drop=True)

# Split Train & Test
last_train_ds_1 = unique_ds_1[-TEST_TIMESTAMPS]
train_df_1 = df_sample_1[df_sample_1['ds'] < last_train_ds_1].copy()
test_df_1 = df_sample_1[df_sample_1['ds'] >= last_train_ds_1].copy()

print(
    f'Informasi Dataset 1 Target:\n'
    f'- Jumlah Fitur Target  : {NUM_TARGETS_1} fitur ({target_col_1[0]})\n'
    f'- Timestep Train Data  : {len(train_df_1) // NUM_TARGETS_1} baris\n'
    f'- Timestep Test Data   : {len(test_df_1) // NUM_TARGETS_1} baris\n'
)

# =====================================================================
# SKENARIO 2: DATASET ALL TARGETS (14 Fitur Cuaca)
# =====================================================================
NUM_TARGETS_ALL = len(all_target)  # 14 Target

# Ambil 10.000 timestamp terakhir berbasis kolom 'ds' untuk SEMUA unique_id
unique_ds_all = df_all['ds'].unique()[-NUM_TIMESTAMPS:]
df_sample_all = df_all[df_all['ds'].isin(unique_ds_all)].reset_index(
    drop=True
)

# Split Train & Test
last_train_ds_all = unique_ds_all[-TEST_TIMESTAMPS]
train_df_all = df_sample_all[df_sample_all['ds'] < last_train_ds_all].copy()
test_df_all = df_sample_all[df_sample_all['ds'] >= last_train_ds_all].copy()

print(
    f'Informasi Dataset All Target:\n'
    f'- Jumlah Fitur Target  : {NUM_TARGETS_ALL} fitur\n'
    f'- Timestep Train Data  : {len(train_df_all) // NUM_TARGETS_ALL} baris per fitur\n'
    f'- Timestep Test Data   : {len(test_df_all) // NUM_TARGETS_ALL} baris per fitur'
)

# **Eksperimen 1: Single Target Benchmark (Fokus 1 Target: Suhu)**

In [ ]:
import os
from neuralforecast import NeuralForecast
from neuralforecast.models import GRU, DLinear, NHiTS, PatchTST, iTransformer

# 1. Inisialisasi Model untuk Eksperimen 1 (Dataset 1 Target)
models_exp1 = [
    iTransformer(
        h=HORIZON,
        input_size=INPUT_SIZE,
        n_series=1,
        futr_exog_list=time_exog_cols,
        max_steps=2000,
        batch_size=32,
        val_check_steps=50,
        early_stop_patience_steps=3,
        scaler_type='standard',
        accelerator='auto',
    ),
    PatchTST(
        h=HORIZON,
        input_size=INPUT_SIZE,
        n_series=1,
        futr_exog_list=time_exog_cols,
        max_steps=2000,
        batch_size=32,
        val_check_steps=50,
        early_stop_patience_steps=3,
        scaler_type='standard',
        accelerator='auto',
    ),
    DLinear(
        h=HORIZON,
        input_size=INPUT_SIZE,
        hist_exog_list=hist_exog_cols_1,
        max_steps=2000,
        batch_size=32,
        val_check_steps=50,
        early_stop_patience_steps=3,
        scaler_type='standard',
        accelerator='auto',
    ),
    NHiTS(
        h=HORIZON,
        input_size=INPUT_SIZE,
        hist_exog_list=hist_exog_cols_1,
        max_steps=2000,
        batch_size=32,
        val_check_steps=50,
        early_stop_patience_steps=3,
        scaler_type='standard',
        accelerator='auto',
    ),
    GRU(
        h=HORIZON,
        input_size=INPUT_SIZE,
        hist_exog_list=hist_exog_cols_1,
        max_steps=2000,
        batch_size=32,
        val_check_steps=50,
        early_stop_patience_steps=3,
        scaler_type='standard',
        accelerator='auto',
    ),
]

# 2. Fit & Save Model Eksperimen 1
nf_exp1 = NeuralForecast(models=models_exp1, freq='h')
print('=== TRARINING EKSPERIMEN 1 (1 TARGET) ===')
nf_exp1.fit(df=train_df_1, val_size=VAL_SIZE)

path_exp1 = './models_exp1'
nf_exp1.save(path=path_exp1, overwrite=True)

# 3. Predict / Cross Validation
cv_exp1 = nf_exp1.cross_validation(
    df=df_sample_1,
    n_windows=TEST_TIMESTAMPS // HORIZON,
    val_size=VAL_SIZE,
    use_fitted=True,
)

# **Eksperimen 2: Optimal Capacity Benchmark (Multivariate vs Exogenous)**

In [ ]:
# 1. Model Multivariate (iTransformer & PatchTST memprediksi 14 Fitur)
models_multi = [
    iTransformer(
        h=HORIZON,
        input_size=INPUT_SIZE,
        n_series=14,
        futr_exog_list=time_exog_cols,
        max_steps=2000,
        batch_size=32,
        val_check_steps=50,
        early_stop_patience_steps=3,
        scaler_type='standard',
        accelerator='auto',
    ),
    PatchTST(
        h=HORIZON,
        input_size=INPUT_SIZE,
        n_series=14,
        futr_exog_list=time_exog_cols,
        max_steps=2000,
        batch_size=32,
        val_check_steps=50,
        early_stop_patience_steps=3,
        scaler_type='standard',
        accelerator='auto',
    ),
]

# 2. Model Univariate dengan Exogenous (DLinear, NHiTS, GRU)
models_exog = [
    DLinear(
        h=HORIZON,
        input_size=INPUT_SIZE,
        hist_exog_list=hist_exog_cols_1,
        max_steps=2000,
        batch_size=32,
        val_check_steps=50,
        early_stop_patience_steps=3,
        scaler_type='standard',
        accelerator='auto',
    ),
    NHiTS(
        h=HORIZON,
        input_size=INPUT_SIZE,
        hist_exog_list=hist_exog_cols_1,
        max_steps=2000,
        batch_size=32,
        val_check_steps=50,
        early_stop_patience_steps=3,
        scaler_type='standard',
        accelerator='auto',
    ),
    GRU(
        h=HORIZON,
        input_size=INPUT_SIZE,
        hist_exog_list=hist_exog_cols_1,
        max_steps=2000,
        batch_size=32,
        val_check_steps=50,
        early_stop_patience_steps=3,
        scaler_type='standard',
        accelerator='auto',
    ),
]

# 3. Fit Kedua Grup Model secara Terpisah
nf_multi = NeuralForecast(models=models_multi, freq='h')
nf_exog = NeuralForecast(models=models_exog, freq='h')

print('=== TRAINING EKSPERIMEN 2: MULTIVARIATE (iTransformer & PatchTST) ===')
nf_multi.fit(df=train_df_all, val_size=VAL_SIZE)

print('\n=== TRAINING EKSPERIMEN 2: EXOGENOUS (DLinear, NHiTS, GRU) ===')
nf_exog.fit(df=train_df_1, val_size=VAL_SIZE)

# 4. Cross Validation
cv_multi = nf_multi.cross_validation(
    df=df_sample_all,
    n_windows=TEST_TIMESTAMPS // HORIZON,
    val_size=VAL_SIZE,
    use_fitted=True,
)
cv_exog = nf_exog.cross_validation(
    df=df_sample_1,
    n_windows=TEST_TIMESTAMPS // HORIZON,
    val_size=VAL_SIZE,
    use_fitted=True,
)

# 5. Gabungkan Hasil Prediksi Khusus Target Suhu 'T (degC)'
cv_multi_suhu = cv_multi[cv_multi['unique_id'] == 'T (degC)']
cv_exp2 = cv_multi_suhu.merge(
    cv_exog[['ds', 'unique_id', 'cutoff', 'DLinear', 'NHiTS', 'GRU']],
    on=['ds', 'unique_id', 'cutoff'],
)

# **membuat plot perbandingan grafik prediksi dari Eksperimen 1 dan Eksperimen 2 khusus target T (degC)**

In [ ]:
import matplotlib.pyplot as plt

# 1. Filter khusus target 'T (degC)' dan ambil 200 timestep terakhir untuk plot
plot_exp1 = cv_exp1[cv_exp1['unique_id'] == 'T (degC)'].tail(200)
plot_exp2 = cv_exp2[cv_exp2['unique_id'] == 'T (degC)'].tail(200)

# Warna konsisten untuk setiap model di kedua subplot
colors = {
    'iTransformer': '#e74c3c',  # Merah
    'PatchTST': '#3498db',  # Biru
    'DLinear': '#2ecc71',  # Hijau
    'NHiTS': '#f39c12',  # Oranye
    'GRU': '#9b59b6',  # Ungu
}

models = ['iTransformer', 'PatchTST', 'DLinear', 'NHiTS', 'GRU']

# 2. Buat Figure dengan 2 Subplot Vertikal
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# --- Subplot 1: Eksperimen 1 (Pure Single Target) ---
ax1.plot(
    plot_exp1['ds'],
    plot_exp1['y'],
    label='Nilai Asli (Actual)',
    color='black',
    linewidth=2,
    alpha=0.85,
)

for model in models:
  if model in plot_exp1.columns:
    ax1.plot(
        plot_exp1['ds'],
        plot_exp1[model],
        label=model,
        color=colors[model],
        linestyle='--',
        alpha=0.75,
    )

ax1.set_title(
    'Eksperimen 1: Single Target Benchmark (Semua Model Hanya Membaca Input'
    ' Suhu)',
    fontsize=11,
    fontweight='bold',
)
ax1.set_ylabel('Suhu T (degC)', fontsize=10)
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.legend(loc='upper left', ncol=3)

# --- Subplot 2: Eksperimen 2 (Optimal Capacity) ---
ax2.plot(
    plot_exp2['ds'],
    plot_exp2['y'],
    label='Nilai Asli (Actual)',
    color='black',
    linewidth=2,
    alpha=0.85,
)

for model in models:
  if model in plot_exp2.columns:
    ax2.plot(
        plot_exp2['ds'],
        plot_exp2[model],
        label=model,
        color=colors[model],
        linestyle='--',
        alpha=0.75,
    )

ax2.set_title(
    'Eksperimen 2: Optimal Capacity (iTransformer & PatchTST 14-Target vs'
    ' DLinear, NHiTS, GRU dengan Exogenous)',
    fontsize=11,
    fontweight='bold',
)
ax2.set_xlabel('Waktu (Timestamp)', fontsize=10)
ax2.set_ylabel('Suhu T (degC)', fontsize=10)
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.legend(loc='upper left', ncol=3)

# Layout adjustment & simpan
plt.suptitle(
    'Komparasi Hasil Prediksi Suhu T (degC): Eksperimen 1 vs Eksperimen 2',
    fontsize=14,
    fontweight='bold',
)
plt.tight_layout()
plt.savefig('komparasi_eksperimen_1_vs_2_suhu.png', dpi=300)
plt.show()